# BIST asistanı: LoRA eğitimi v2, devam (Kaggle, ücretsiz T4)

Colab'ın ücretsiz GPU kotası dolduğu için eğitim burada, Drive'daki son kayıttan (`state.json`) sürer. Betik ve veri Colab'dakiyle aynı.

**Ayarlar** (sağdaki panel → *Session options*):
1. **Accelerator: GPU T4 x2** (P100 değil; Colab'daki T4 ile aynı GPU, yalnızca birincisi kullanılır).
2. **Internet: On** (model Hugging Face'ten indirilir, ~8 GB; telefon doğrulaması gerekir).
3. **Input**: *Add Input* → kendi `bist-v2` veri setin.

**Çalıştırma**: sağ üstte **Save Version → Save & Run All (Commit) → Save**. Defter arka planda çalışır:
tarayıcıyı kapatabilirsin, internet kopsa da durmaz. İlerleme: sürüme tıkla → **Logs**.
Bitince sürümün **Output** sekmesinden `adapter_v2.zip` dosyasını indir (~60 MB).


### GPU, dosyalar ve son kayıt

In [ ]:
import glob, gzip, hashlib, json, os, shutil, zipfile
import torch
assert torch.cuda.is_available(), "GPU yok: sağ panel > Session options > Accelerator > GPU T4 x2 seç."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

IN, W, TMP = "/kaggle/input", "/kaggle/working", "/tmp/bist_unzip"
for z in glob.glob(f"{IN}/**/*.zip", recursive=True):        # zip açılmadan yüklendiyse
    zipfile.ZipFile(z).extractall(TMP)

def find(name):
    hits = glob.glob(f"{IN}/**/{name}", recursive=True) or glob.glob(f"{TMP}/**/{name}", recursive=True)
    assert hits, f"{name} yok: veri setini yükleyip sağ panelde Add Input ile eklediğinden emin ol."
    if len({hashlib.md5(open(h, "rb").read()).hexdigest() for h in hits}) > 1:
        raise AssertionError(f"{name} farklı içerikle birden fazla yerde: {hits}")
    return hits[0]

A = f"{W}/adapter_v2"
os.makedirs(A, exist_ok=True)
if not os.path.exists(f"{A}/state.json"):                    # her sürüm boş başlar: Drive kaydını kopyala
    for f in ("adapter_config.json", "adapter_model.safetensors", "state.json"):
        shutil.copy(find(f), A)
    for f in ("train_gpu.py", "train_v2_log.jsonl"):
        shutil.copy(find(f), W)
    try:
        shutil.copy(find("tokenized.jsonl.gz"), W)
    except AssertionError:                                   # Kaggle .gz'yi açtıysa yeniden sıkıştır
        with open(find("tokenized.jsonl"), "rb") as s, gzip.open(f"{W}/tokenized.jsonl.gz", "wb") as d:
            shutil.copyfileobj(s, d)
st = json.load(open(f"{A}/state.json"))
print("son kayıt:", st["micro_step"], "/", st["of"], "| son konuşma:", st["last_conversation"])
assert 0 < st["micro_step"] < st["of"], "state.json beklenen gibi değil"


### Kütüphaneler (Mac ve Colab'daki sürümlerle aynı)
Kırmızı *dependency conflicts* uyarıları normal, Kaggle'ın başka paketleriyle ilgili.

In [ ]:
%pip install -q "transformers==5.24.0" "peft==0.22.0" "accelerate==1.16.0"
import subprocess, sys
subprocess.run([sys.executable, "-c", "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"])


### Eğitim (kalan kısım: model indirme dahil tahmini ~1 saat)
`run_train.py`, `train_gpu.py`'yi değiştirmeden çalıştırır; tek farkı dikkat (SDPA) hesabını bellek-verimli çekirdekte tutmak
(Kaggle'ın torch 2.11'inde GQA yolu T4'te dikkat matrisini tümüyle kurup belleği taşırdı: 4.621 tokende OOM; Colab'da 8.946 token 13,1 GB'a sığmıştı).
Önce `SDPA denemesi tamam` satırı, sonra `start_micro_step` = son kayıt (0 değil). Her 5 adımda bir satır: `eta_min` = kalan dakika.

In [ ]:
RUN_SRC = '"""train_gpu.py\'yi aynen çalıştırır; yalnızca dikkat (SDPA) bellek-verimli çekirdekte kalır."""\nimport runpy, sys, torch\nimport transformers.integrations.sdpa_attention as sa\n\nsa.use_gqa_in_sdpa = lambda *a, **k: False       # GQA: enable_gqa yerine K/V tekrarı (repeat_kv)\ntorch.backends.cuda.enable_math_sdp(False)        # dikkat matrisini tümüyle kuran çekirdek kapalı: sorun varsa hemen hata\nq = torch.randn(1, 32, 8192, 128, device="cuda", dtype=torch.float16, requires_grad=True)\ntorch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True).sum().backward()\nprint(f"SDPA denemesi tamam (8.192 token, ileri+geri): tepe bellek {torch.cuda.max_memory_allocated() / 1e9:.2f} GB", flush=True)\ndel q\ntorch.cuda.empty_cache()\ntorch.cuda.reset_peak_memory_stats()\nsys.argv = ["train_gpu.py"] + sys.argv[1:]\nrunpy.run_path("train_gpu.py", run_name="__main__")\n'
open(f"{W}/run_train.py", "w").write(RUN_SRC)
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
p = subprocess.Popen([sys.executable, "run_train.py", "train", "--resume", "--out", "adapter_v2",
                      "--log", "train_v2_log.jsonl"], cwd=W, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in p.stdout:
    print(line, end="", flush=True)
rc = p.wait()
print("çıkış kodu:", rc)


### Paketle
Eğitim yarıda kalsa bile son kayıt zip'e girer (oradan yine sürdürülebilir).

In [ ]:
with zipfile.ZipFile(f"{W}/adapter_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{A}/*")):
        z.write(f, "adapter_v2/" + os.path.basename(f))
    z.write(f"{W}/train_v2_log.jsonl", "train_v2_log.jsonl")
print("adapter_v2.zip", round(os.path.getsize(f"{W}/adapter_v2.zip") / 1e6, 1), "MB")
last = open(f"{W}/train_v2_log.jsonl").read().strip().splitlines()[-1]
print("son log satırı:", last)
assert rc == 0 and json.loads(last)["kind"] == "done", "eğitim bitmedi; zip yine de Output'ta (son kayıt dahil)"
